# Sorted Set examples

Use sorted sets to keep members ordered by score, query ranges, and build a leaderboard.

## Connect to Redis

Run a Redis server before executing this notebook. Set `REDIS_URL` to use a server other than `localhost:6379`.

With `decode_responses=True`, members are returned as Python strings; scores are returned as floats. See the [connection examples](connection_examples.ipynb) for more details.

Each run uses a unique key prefix. The final section removes only the keys created by this notebook.

In [1]:
import os
from uuid import uuid4

import redis

r = redis.Redis.from_url(
    os.environ.get("REDIS_URL", "redis://localhost:6379/0"),
    decode_responses=True,
)

prefix = f"redis-py:sorted-set-example:{uuid4().hex}"
scores_key = f"{prefix}:scores"
round1_key = f"{prefix}:round1"
round2_key = f"{prefix}:round2"
total_key = f"{prefix}:total"
common_key = f"{prefix}:common"
leaderboard_key = f"{prefix}:leaderboard"

r.ping()

True

## Basic ```ZADD``` and ```ZRANGE```

```ZADD``` adds members with scores to a sorted set. Members are automatically ordered by score.

In [2]:
r.delete(scores_key)

r.zadd(scores_key, {"alice": 100, "bob": 200, "charlie": 150})

3

```ZRANGE``` returns members ordered by score (ascending by default)

In [3]:
r.zrange(scores_key, 0, -1)

['alice', 'charlie', 'bob']

Pass ```withscores=True``` to include scores in the result

In [4]:
r.zrange(scores_key, 0, -1, withscores=True)

[('alice', 100.0), ('charlie', 150.0), ('bob', 200.0)]

## ```ZRANGEBYSCORE```

Query members whose scores fall within a given range

In [5]:
r.zrangebyscore(scores_key, 100, 150, withscores=True)

[('alice', 100.0), ('charlie', 150.0)]

Use ```"-inf"``` and ```"+inf"``` for unbounded ranges

In [6]:
r.zrangebyscore(scores_key, "-inf", 150, withscores=True)

[('alice', 100.0), ('charlie', 150.0)]

## ```ZINCRBY```

Increment the score of a member. This is useful for updating scores in real time.

In [7]:
r.zincrby(scores_key, 75, "alice")

175.0

In [8]:
r.zrange(scores_key, 0, -1, withscores=True)

[('charlie', 150.0), ('alice', 175.0), ('bob', 200.0)]

## ```ZRANK``` and ```ZREVRANK```

```ZRANK``` returns the rank of a member (0-based, lowest score first). ```ZREVRANK``` returns the rank with the highest score first.

In [9]:
r.zrank(scores_key, "bob")

2

In [10]:
r.zrevrank(scores_key, "bob")

0

## ```ZREM```

Remove one or more members from a sorted set

In [11]:
r.zrem(scores_key, "charlie")

1

In [12]:
r.zrange(scores_key, 0, -1, withscores=True)

[('alice', 175.0), ('bob', 200.0)]

## ```ZUNIONSTORE``` and ```ZINTERSTORE```

Combine multiple sorted sets. ```ZUNIONSTORE``` stores the union and ```ZINTERSTORE``` stores the intersection of the input sets.

In [13]:
r.delete(round1_key, round2_key, total_key, common_key)

r.zadd(round1_key, {"alice": 100, "bob": 200})
r.zadd(round2_key, {"bob": 150, "charlie": 300})

2

Union sums scores by default for members that appear in both sets

In [14]:
r.zunionstore(total_key, [round1_key, round2_key])
r.zrange(total_key, 0, -1, withscores=True)

[('alice', 100.0), ('charlie', 300.0), ('bob', 350.0)]

Intersection keeps only members present in all input sets

In [15]:
r.zinterstore(common_key, [round1_key, round2_key])
r.zrange(common_key, 0, -1, withscores=True)

[('bob', 350.0)]

## Practical example: Leaderboard

Sorted sets are a natural fit for leaderboards. Scores are kept in order automatically, making rank lookups very efficient.

In [16]:
r.delete(leaderboard_key)

# Add players with their scores
r.zadd(
    leaderboard_key,
    {
        "player:anna": 2500,
        "player:ben": 4000,
        "player:cara": 3500,
        "player:dan": 1800,
        "player:eve": 4200,
    },
)

5

In [17]:
# Top 3 players (highest scores first)
r.zrevrange(leaderboard_key, 0, 2, withscores=True)

[('player:eve', 4200.0), ('player:ben', 4000.0), ('player:cara', 3500.0)]

In [18]:
# A player completes a level and earns bonus points
r.zincrby(leaderboard_key, 600, "player:anna")

# Check anna's new rank (0-based, highest score first)
r.zrevrank(leaderboard_key, "player:anna")

3

In [19]:
# Get a specific player's score
r.zscore(leaderboard_key, "player:anna")

3100.0

In [20]:
# Total number of players on the leaderboard
r.zcard(leaderboard_key)

5

In [21]:
# Final leaderboard (highest to lowest)
r.zrevrange(leaderboard_key, 0, -1, withscores=True)

[('player:eve', 4200.0),
 ('player:ben', 4000.0),
 ('player:cara', 3500.0),
 ('player:anna', 3100.0),
 ('player:dan', 1800.0)]

## Cleanup

Remove the example keys and close the connection. Other keys in the database are left untouched.

In [22]:
r.delete(scores_key, round1_key, round2_key, total_key, common_key, leaderboard_key)
r.close()